# Metric 3 — BGP Topology (T1): IPv6 AS-Level Topology Analysis

**TDDE35 Group 1 — David**

Replicates and extends the T1 metric from Czyz et al. (2015) using RouteViews BGP snapshots.

**What this notebook measures:**
- Number of ASes advertising IPv6 prefixes over time (2004–2025)
- Number of unique IPv6 prefixes in the routing table
- Hurricane Electric (AS 6939) presence in IPv6 AS-paths
- Comparison of IPv4 vs IPv6 routing table growth

## Google Colab Setup

**Run this section first when opening in Colab.**  
It mounts your Google Drive so that downloaded files and parsed results are saved there — meaning you never re-download across sessions.

In [ ]:
import os

IN_COLAB = 'COLAB_JUPYTER_IP' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # All cache lives in your Drive — survives across sessions
    BASE_DIR   = '/content/drive/MyDrive/bgp_metric3'
    CACHE_DIR  = BASE_DIR + '/bgp_cache'
    RIB_DIR    = BASE_DIR + '/rib_files'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(RIB_DIR,   exist_ok=True)
    print('Running in Colab — cache stored in Google Drive at', BASE_DIR)
else:
    # Local machine paths (unchanged)
    CACHE_DIR = './bgp_cache'
    RIB_DIR   = './rib_files'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(RIB_DIR,   exist_ok=True)
    print('Running locally — cache stored in', CACHE_DIR)

## 0. Install dependencies

In [2]:
# Install required packages
!pip install mrtparse requests matplotlib pandas tqdm


## 1. Imports and Configuration

In [ ]:
import mrtparse
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm.notebook import tqdm
import json
import os
import time

# ── Configuration ─────────────────────────────────────────────────────────────
COLLECTORS = {
    'route-views2': 'https://archive.routeviews.org/bgpdata/',
    'route-views6': 'https://archive.routeviews.org/route-views6/bgpdata/',
}

SAMPLE_YEARS = list(range(2004, 2026))
SAMPLE_MONTH = '01'
HE_ASN = 6939

# CACHE_DIR and RIB_DIR are set in the Colab Setup cell above.
# If running locally without that cell, set them here as fallback:
if 'CACHE_DIR' not in dir():
    CACHE_DIR = './bgp_cache'
    RIB_DIR   = './rib_files'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(RIB_DIR,   exist_ok=True)

print('Configuration loaded.')
print(f'Analysing years: {SAMPLE_YEARS[0]} – {SAMPLE_YEARS[-1]}')
print(f'Cache dir: {CACHE_DIR}')

## 2. Helper — Build RouteViews RIB URL

RouteViews stores RIB (Routing Information Base) dumps at predictable URLs:
```
https://archive.routeviews.org/bgpdata/YYYY.MM/RIBS/rib.YYYYMMDD.HHMM.bz2
```

In [4]:
def build_rib_url(year: int, month: str, day: str, collector: str = 'route-views2') -> str:
    """
    Build the URL for a RouteViews RIB snapshot.
    Tries 0000 hours first; RouteViews dumps at 0000, 0800, 1600 UTC.
    """
    base = COLLECTORS[collector]
    ym = f'{year}.{month}'
    date_str = f'{year}{month}{day}'
    return f'{base}{ym}/RIBS/rib.{date_str}.0000.bz2'


def url_exists(url: str) -> bool:
    """HEAD request to check if a file exists on RouteViews."""
    try:
        r = requests.head(url, timeout=10)
        return r.status_code == 200
    except Exception:
        return False


def find_rib_url(year: int, collector: str = 'route-views2') -> str | None:
    """
    Try a few days/times in January to find a valid RIB dump.
    Returns the first URL that exists, or None.
    """
    for day in ['02', '01', '03', '08']:
        for hour in ['0000', '0800', '1600']:
            base = COLLECTORS[collector]
            ym = f'{year}.{SAMPLE_MONTH}'
            date_str = f'{year}{SAMPLE_MONTH}{day}'
            url = f'{base}{ym}/RIBS/rib.{date_str}.{hour}.bz2'
            if url_exists(url):
                return url
    return None

# Quick test
test_url = find_rib_url(2023, 'route-views2')
print(f'Sample 2023 URL: {test_url}')

Sample 2023 URL: https://archive.routeviews.org/bgpdata/2023.01/RIBS/rib.20230102.0000.bz2


## 3. Parse a Single RIB Snapshot with bgpkit-parser

For each snapshot we extract:
- All **IPv6 prefixes** (prefix contains `:`)
- All **originating ASes** (last AS in the AS-path)
- All **AS-paths** containing HE (AS 6939)

In [ ]:
import urllib.request

def download_rib(url: str, year: int, collector: str) -> str:
    """Download RIB file if not already cached locally. Returns local path."""
    local_path = os.path.join(RIB_DIR, f'{year}_{collector}.bz2')
    if os.path.exists(local_path):
        print(f'  Using cached file for {year} ({collector})')
        return local_path
    print(f'  Downloading {year} from {collector}...')
    urllib.request.urlretrieve(url, local_path)
    print(f'  Saved to {local_path} ({os.path.getsize(local_path)/1e6:.0f} MB)')
    return local_path

HE_ASN_STR = str(HE_ASN)

def parse_ipv6_snapshot(url: str, year: int, collector: str = 'route-views6') -> dict:
    """Parse IPv6 stats from a RouteViews collector. Cached as {year}_{collector}_ipv6_stats.json.

    HE fraction is computed *per prefix* (does any path for this prefix contain HE?),
    matching the Czyz et al. definition.  The old per-path fraction is also stored for
    reference as ipv6_he_path_fraction.
    """
    cache_file = os.path.join(CACHE_DIR, f'{year}_{collector}_ipv6_stats.json')

    # Load cache, but invalidate if it pre-dates the per-prefix HE field.
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            cached = json.load(f)
        if 'ipv6_he_prefix_fraction' in cached:
            return cached
        print(f'  Cache for {year} {collector} is outdated — recomputing...')
        os.remove(cache_file)

    stats = {
        'ipv6_prefixes':     set(),
        'ipv6_he_prefixes':  set(),   # prefixes where ≥1 path contains HE
        'ipv6_origin_ases':  set(),
        'ipv6_total_paths':  0,
        'ipv6_he_paths':     0,
    }
    try:
        local_path = download_rib(url, year, collector)
        print(f'  Parsing IPv6 {year} ({collector})...')
        n = 0
        for entry in mrtparse.Reader(local_path):
            if entry.err:
                continue
            subtype_val = list(entry.data.get('subtype', {0: ''}).keys())[0]
            if subtype_val != 4:   # RIB_IPV6_UNICAST in TABLE_DUMP_V2
                continue
            prefix = f"{entry.data.get('prefix','')}/{entry.data.get('prefix_length',0)}"
            stats['ipv6_prefixes'].add(prefix)
            prefix_has_he = False
            for rib in entry.data.get('rib_entries', []):
                as_path = []
                for attr in rib.get('path_attributes', []):
                    if list(attr.get('type', {0: ''}).keys())[0] == 2:
                        for seg in attr.get('value', []):
                            as_path += seg.get('value', [])
                origin_as = as_path[-1] if as_path else None
                stats['ipv6_total_paths'] += 1
                if origin_as:
                    stats['ipv6_origin_ases'].add(origin_as)
                if HE_ASN_STR in as_path:
                    stats['ipv6_he_paths'] += 1
                    prefix_has_he = True
            if prefix_has_he:
                stats['ipv6_he_prefixes'].add(prefix)
            n += 1
            if n % 50000 == 0:
                print(f'  {n:,} entries...', end='\r')
    except Exception as e:
        print(f'  [!] IPv6 error {year} ({collector}): {e}')
        return None

    total_prefixes = len(stats['ipv6_prefixes'])
    total_paths    = stats['ipv6_total_paths']
    result = {
        'collector':               collector,
        'ipv6_prefix_count':       total_prefixes,
        'ipv6_origin_as_count':    len(stats['ipv6_origin_ases']),
        'ipv6_total_paths':        total_paths,
        'ipv6_he_paths':           stats['ipv6_he_paths'],
        # Per-prefix fraction — matches Czyz et al. definition
        'ipv6_he_prefix_count':    len(stats['ipv6_he_prefixes']),
        'ipv6_he_prefix_fraction': (len(stats['ipv6_he_prefixes']) / total_prefixes
                                    if total_prefixes > 0 else 0),
        # Per-path fraction — kept for reference
        'ipv6_he_path_fraction':   (stats['ipv6_he_paths'] / total_paths
                                    if total_paths > 0 else 0),
    }
    with open(cache_file, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'\n  IPv6 cached for {year} ({collector})')
    return result

def parse_ipv4_snapshot(url: str, year: int) -> dict:
    """Parse IPv4 stats from route-views2. Cached as {year}_ipv4_stats.json."""
    cache_file = os.path.join(CACHE_DIR, f'{year}_ipv4_stats.json')
    if os.path.exists(cache_file):
        with open(cache_file) as f:
            return json.load(f)

    stats = {'ipv4_prefixes': set(), 'ipv4_origin_ases': set()}
    try:
        local_path = download_rib(url, year, 'route-views2')
        print(f'  Parsing IPv4 {year}...')
        n = 0
        for entry in mrtparse.Reader(local_path):
            if entry.err:
                continue
            subtype_val = list(entry.data.get('subtype', {0: ''}).keys())[0]
            if subtype_val != 2:   # RIB_IPV4_UNICAST
                continue
            prefix = f"{entry.data.get('prefix','')}/{entry.data.get('prefix_length',0)}"
            for rib in entry.data.get('rib_entries', []):
                as_path = []
                for attr in rib.get('path_attributes', []):
                    if list(attr.get('type', {0: ''}).keys())[0] == 2:
                        for seg in attr.get('value', []):
                            as_path += seg.get('value', [])
                origin_as = as_path[-1] if as_path else None
                stats['ipv4_prefixes'].add(prefix)
                if origin_as:
                    stats['ipv4_origin_ases'].add(origin_as)
            n += 1
            if n % 50000 == 0:
                print(f'  {n:,} entries...', end='\r')
    except Exception as e:
        print(f'  [!] IPv4 error {year}: {e}')
        return None

    result = {
        'ipv4_prefix_count':    len(stats['ipv4_prefixes']),
        'ipv4_origin_as_count': len(stats['ipv4_origin_ases']),
    }
    with open(cache_file, 'w') as f:
        json.dump(result, f, indent=2)
    print(f'\n  IPv4 cached for {year}')
    return result

print('parse_ipv6_snapshot() and parse_ipv4_snapshot() defined.')


## 4. Run the Historical Collection (2004–2025)

⚠️ **This cell takes time** — each RIB file is ~1–4 GB compressed.
Results are cached in `./bgp_cache/` so re-runs are instant.

Tip: start with a subset (e.g. `SAMPLE_YEARS[-5:]`) to verify it works,
then run the full range overnight.

In [ ]:
## Run full historical collection (2004–2025)
# IPv6: prefer route-views6; fall back to route-views2 for early years (2004-2008)
#        when route-views6 did not yet exist.
# IPv4: always from route-views2.
rows = []
for year in tqdm(SAMPLE_YEARS, desc='Years'):
    url6 = find_rib_url(year, 'route-views6')
    url4 = find_rib_url(year, 'route-views2')

    # Determine IPv6 source
    if url6:
        ipv6 = parse_ipv6_snapshot(url6, year, collector='route-views6')
    elif url4:
        # route-views6 didn't exist yet — parse IPv6 routes from route-views2
        print(f'  [route-views6 unavailable for {year}] falling back to route-views2 for IPv6')
        ipv6 = parse_ipv6_snapshot(url4, year, collector='route-views2')
    else:
        ipv6 = None

    ipv4 = parse_ipv4_snapshot(url4, year) if url4 else None

    if ipv6 is None and ipv4 is None:
        print(f'[!] No data for {year}, skipping.')
        continue

    row = {'year': year}
    row.update(ipv6 or {
        'ipv6_prefix_count': 0, 'ipv6_origin_as_count': 0,
        'ipv6_total_paths': 0, 'ipv6_he_paths': 0,
        'ipv6_he_prefix_count': 0, 'ipv6_he_prefix_fraction': 0,
        'ipv6_he_path_fraction': 0,
        'collector': 'none',
    })
    row.update(ipv4 or {'ipv4_prefix_count': 0, 'ipv4_origin_as_count': 0})
    rows.append(row)

df = pd.DataFrame(rows).sort_values('year').reset_index(drop=True)
df['ipv6_share_prefixes'] = (
    df['ipv6_prefix_count'] / (df['ipv6_prefix_count'] + df['ipv4_prefix_count']) * 100
).fillna(0)

print(f'\nDone. {len(df)} years collected.')
print(df[['year', 'collector', 'ipv6_prefix_count', 'ipv4_prefix_count',
          'ipv6_he_prefix_fraction', 'ipv6_he_path_fraction']].to_string(index=False))


## 5. Results Table

In [ ]:
display_cols = [
    'year', 'collector',
    'ipv6_prefix_count', 'ipv4_prefix_count',
    'ipv6_origin_as_count', 'ipv4_origin_as_count',
    'ipv6_he_prefix_fraction',   # per-prefix (Czyz definition)
    'ipv6_he_path_fraction',     # per-path (for reference)
]

df_display = df[display_cols].copy()
df_display['ipv6_he_prefix_fraction'] = df_display['ipv6_he_prefix_fraction'].map('{:.1%}'.format)
df_display['ipv6_he_path_fraction']   = df_display['ipv6_he_path_fraction'].map('{:.1%}'.format)
df_display.columns = [
    'Year', 'Collector',
    'IPv6 Prefixes', 'IPv4 Prefixes',
    'IPv6 Origin ASes', 'IPv4 Origin ASes',
    'HE in ≥1 path (per-prefix)',
    'HE paths / total paths (per-path)',
]
df_display


## 5b. Load results from CSV (skip collection)

Run this cell instead of section 4 if you already have a saved CSV and just want to regenerate plots.

In [ ]:
import pandas as pd, os

# Try local path first, then Colab Drive path.
_candidates = [
    'metric3_bgp_results.csv',
    '../out/metric3_bgp_results.csv',
    '/content/drive/MyDrive/bgp_metric3/metric3_bgp_results.csv',
]
_csv = next((p for p in _candidates if os.path.exists(p)), None)
assert _csv, f'No CSV found — tried: {_candidates}'

df = pd.read_csv(_csv)

# ── Column compatibility: handle old (per-path) and new (per-prefix) exports ──
if 'ipv6_he_prefix_fraction' not in df.columns:
    # Old CSV only has the per-path metric under the name ipv6_he_fraction.
    # Rename it for the plot cells; per-prefix data is not available from this export.
    df.rename(columns={'ipv6_he_fraction': 'ipv6_he_path_fraction'}, inplace=True)
    df['ipv6_he_prefix_fraction'] = float('nan')
    print('⚠ Loaded OLD CSV — ipv6_he_prefix_fraction not available; per-prefix plot will be empty.')
else:
    print('Loaded new CSV with per-prefix HE fraction.')

if 'ipv6_he_path_fraction' not in df.columns and 'ipv6_he_fraction' in df.columns:
    df.rename(columns={'ipv6_he_fraction': 'ipv6_he_path_fraction'}, inplace=True)

if 'ipv6_share_prefixes' not in df.columns:
    df['ipv6_share_prefixes'] = (
        df['ipv6_prefix_count'] / (df['ipv6_prefix_count'] + df['ipv4_prefix_count']) * 100
    ).fillna(0)

if 'collector' not in df.columns:
    df['collector'] = 'unknown'

df = df.sort_values('year').reset_index(drop=True)

# Drop years with no meaningful data (both counts zero)
df = df[(df['ipv6_prefix_count'] > 0) | (df['ipv4_prefix_count'] > 0)].reset_index(drop=True)

print(f'Loaded {len(df)} rows from {_csv}')
print(df[['year', 'ipv6_prefix_count', 'ipv4_prefix_count',
          'ipv6_he_path_fraction', 'ipv6_he_prefix_fraction']].to_string(index=False))


## 6. Plots

### 6a. IPv6 vs IPv4 Prefix Growth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

LABEL_FS  = 14
TITLE_FS  = 15
TICK_FS   = 12
LEGEND_FS = 12

# ── Left: Prefix counts ───────────────────────────────────────────────────────
ax = axes[0]
ax.plot(df['year'], df['ipv6_prefix_count'], 'o-', color='steelblue',
        label='IPv6 prefixes', linewidth=2, markersize=5)
ax2 = ax.twinx()
ax2.plot(df['year'], df['ipv4_prefix_count'], 's--', color='coral',
         label='IPv4 prefixes', linewidth=2, markersize=5)

ax.set_xlabel('Year', fontsize=LABEL_FS)
ax.set_ylabel('IPv6 Prefix Count', color='steelblue', fontsize=LABEL_FS)
ax2.set_ylabel('IPv4 Prefix Count', color='coral', fontsize=LABEL_FS)
ax.set_title('IPv6 vs IPv4 Prefix Count (RouteViews)', fontsize=TITLE_FS)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax2.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(labelsize=TICK_FS)
ax2.tick_params(labelsize=TICK_FS)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=LEGEND_FS)
ax.grid(alpha=0.3)

# ── Right: Origin AS counts ───────────────────────────────────────────────────
ax = axes[1]
ax.plot(df['year'], df['ipv6_origin_as_count'], 'o-', color='steelblue',
        label='IPv6 origin ASes', linewidth=2, markersize=5)
ax3 = ax.twinx()
ax3.plot(df['year'], df['ipv4_origin_as_count'], 's--', color='coral',
         label='IPv4 origin ASes', linewidth=2, markersize=5)

ax.set_xlabel('Year', fontsize=LABEL_FS)
ax.set_ylabel('IPv6 Origin AS Count', color='steelblue', fontsize=LABEL_FS)
ax3.set_ylabel('IPv4 Origin AS Count', color='coral', fontsize=LABEL_FS)
ax.set_title('IPv6 vs IPv4 Origin AS Count', fontsize=TITLE_FS)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax3.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(labelsize=TICK_FS)
ax3.tick_params(labelsize=TICK_FS)
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=LEGEND_FS)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metric3_prefix_as_growth.pdf', bbox_inches='tight')
plt.savefig('metric3_prefix_as_growth.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved metric3_prefix_as_growth.pdf')

### 6b. Hurricane Electric (AS 6939) Dominance in IPv6 AS-Paths

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Per-prefix fraction — matches Czyz et al. definition
he_pct = df['ipv6_he_prefix_fraction'] * 100

ax.fill_between(df['year'], he_pct, alpha=0.2, color='darkorange')
ax.plot(df['year'], he_pct, 'o-', color='darkorange', linewidth=2, markersize=6,
        label='AS 6939 (HE) — % of prefixes with HE in ≥1 path')

# Also show per-path fraction as a dashed secondary line for reference
ax.plot(df['year'], df['ipv6_he_path_fraction'] * 100, 's--', color='goldenrod',
        linewidth=1.5, markersize=4, alpha=0.7,
        label='AS 6939 (HE) — % of all paths containing HE (per-path)')

# Czyz et al. 2013 reference (~95% per-prefix)
ax.axhline(y=95, color='red', linestyle='--', linewidth=1, alpha=0.7,
           label='Czyz et al. 2013 baseline (~95%, per-prefix)')

ax.set_xlabel('Year')
ax.set_ylabel('% of IPv6 prefixes/paths containing HE (AS 6939)')
ax.set_title('Hurricane Electric Dominance in IPv6 BGP Topology')
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metric3_he_dominance.pdf', bbox_inches='tight')
plt.savefig('metric3_he_dominance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved metric3_he_dominance.pdf')


### 6c. IPv6 Share of Total Routing Table

In [ ]:
df['ipv6_share_prefixes'] = (
    df['ipv6_prefix_count'] /
    (df['ipv6_prefix_count'] + df['ipv4_prefix_count']) * 100
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(df['year'], df['ipv6_share_prefixes'], alpha=0.15, color='steelblue')
ax.plot(df['year'], df['ipv6_share_prefixes'], 'o-', color='steelblue',
        linewidth=2, markersize=6)

ax.set_xlabel('Year')
ax.set_ylabel('IPv6 share of total routing table (%)')
ax.set_title('IPv6 Share of BGP Routing Table (RouteViews)')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('metric3_ipv6_share.pdf', bbox_inches='tight')
plt.savefig('metric3_ipv6_share.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved metric3_ipv6_share.pdf')

## 7. Save Results to CSV

In [ ]:
df.to_csv('metric3_bgp_results.csv', index=False)
print('Results saved to metric3_bgp_results.csv')
print(df[['year', 'collector', 'ipv6_prefix_count', 'ipv6_origin_as_count',
          'ipv6_he_prefix_fraction', 'ipv6_he_path_fraction',
          'ipv4_prefix_count']].to_string(index=False))


## 8. Quick Sanity Check — Single Recent Year

Run this first to verify the pipeline works before the full 2004–2025 run.

In [ ]:
# Sanity check: parse 2023 from both collectors
test_year = 2023
url6 = find_rib_url(test_year, 'route-views6')
url4 = find_rib_url(test_year, 'route-views2')
print(f'IPv6 URL for {test_year}: {url6}')
print(f'IPv4 URL for {test_year}: {url4}')

ipv6_stats = {}
ipv4_stats = {}

if url6:
    ipv6_stats = parse_ipv6_snapshot(url6, test_year, collector='route-views6')
if url4:
    ipv4_stats = parse_ipv4_snapshot(url4, test_year)

if ipv6_stats or ipv4_stats:
    print(f"\nIPv6 prefixes:                {ipv6_stats.get('ipv6_prefix_count', 0):,}")
    print(f"IPv6 origin ASes:             {ipv6_stats.get('ipv6_origin_as_count', 0):,}")
    print(f"HE in ≥1 path (per-prefix):  {ipv6_stats.get('ipv6_he_prefix_fraction', 0):.1%}  ← compare to Czyz ~95%")
    print(f"HE paths / total (per-path): {ipv6_stats.get('ipv6_he_path_fraction', 0):.1%}  ← old (diluted) metric")
    print(f"IPv4 prefixes:                {ipv4_stats.get('ipv4_prefix_count', 0):,}")
    print(f"IPv4 origin ASes:             {ipv4_stats.get('ipv4_origin_as_count', 0):,}")
